In [1]:
import sys
import os
from itertools import chain
import pandas as pd
import json

sys.path.append('../')

from analysis_utils import prepare_data
from data_utils import parse_json_string, parse_annotation_dict

RESTRICT_TO_VALID = True

REF_ANN_PATH = os.path.realpath(
    "../color-grid-segment-annotations/data/processed_annotations/consensus_majority_clean.json"
)  # preprocessed annotations for this project
COLORGRID_ANN_PATH = os.path.realpath(
    "../color-grid-segment-annotations/data/color_grid_data.json"
)  # original annotations (formatted)
LS_DATA_PATH = os.path.realpath(
    "../color-grid-segment-annotations/data/label_studio_input/label_studio_input.json"
)  # label studio inputs

MODEL_ANNOTATION_DIR = os.path.realpath('../output/generated_annotations')

In [2]:
ann_df, data_df, gameids, (far_ids, split_ids, close_ids) = prepare_data(REF_ANN_PATH, COLORGRID_ANN_PATH, LS_DATA_PATH)

read ann data...
read source data...
add further information from label studio data...


In [3]:
if RESTRICT_TO_VALID:
    print('restrict to valid annotations')
    ann_df = ann_df[ann_df.label_set.map(len) > 1]

restrict to valid annotations


In [4]:
_data_df = data_df.loc[data_df.round_id.isin(ann_df.round_id)]

print('Success rates:')

print('Micro average:')
print((_data_df.success.mean()* 100).round(1), '%')

print('Per Game stats:')
(_data_df.groupby('gameid').success.mean()*100).describe().round(2)

Success rates:
Micro average:
95.0 %
Per Game stats:


count     25.00
mean      95.01
std        4.10
min       84.48
25%       93.22
50%       96.61
75%       98.31
max      100.00
Name: success, dtype: float64

# Data Preparation

In [5]:
SELECTED_COLUMNS = [
    "game_id",
    "round_num",
    "round_id",
    "start",
    "end",
    "span_text",
    "full_text",
    "label_set",
    "ann_url",
    "condition",
]

ann_df_reduced = ann_df[SELECTED_COLUMNS]
ann_df_reduced["span"] = ann_df_reduced.apply(lambda x: (x.start, x.end), axis=1)
ann_df_reduced = ann_df_reduced.drop(columns=["start", "end"])
ann_df_reduced.label_set = ann_df_reduced.label_set.map(
    lambda label_set: [x for x in label_set if x != "with_reference"]
)
ann_df_reduced["n_utterances"] = ann_df.round_id.map(data_df.set_index("round_id").n_utterances)

In [6]:
collapsed_df = (
    ann_df_reduced.groupby("round_id")
    .agg(
        {
            c: (
                "first"
                if c
                in [
                    "game_id",
                    "round_num",
                    "round_id",
                    "full_text",
                    "ann_url",
                    "n_utterances",
                    "condition",
                ]
                else list
            )
            for c in ann_df_reduced.columns
        }
    )
    .sort_values(by=["game_id", "round_num"])
)

collapsed_df["unique_labels"] = collapsed_df.label_set.apply(lambda x: set(chain(*x)))
collapsed_df["n_unique_labels"] = collapsed_df.unique_labels.map(len)
collapsed_df["includes_distractor_reference"] = collapsed_df.unique_labels.map(lambda x: any([label.startswith('d') for label in x]))
collapsed_df["n_spans"] = collapsed_df.label_set.map(len)

# Parse Model Annotations

In [7]:
file = "Qwen36-35B-A3B-FP8_full_human_annotations.json"  # file containing model annotations for human descriptions

with open(os.path.join(MODEL_ANNOTATION_DIR, file), "r") as f:
    model_annotations = json.load(f)
    
    if isinstance(model_annotations, dict) and  "responses" in model_annotations.keys():
        model_annotations_df = pd.DataFrame(model_annotations["responses"])
    else:
        model_annotations_df = pd.DataFrame(model_annotations)

In [8]:
# extract info from model annotations
model_annotations_df['json_response'] = model_annotations_df.model_response.map(parse_json_string)
model_annotations_df[['n_spans', 'spans', 'span_texts', 'cells', 'unique_cells', 'comments', 'no_reference']] = model_annotations_df.json_response.apply(lambda x: pd.Series(parse_annotation_dict(x)))
model_annotations_df = model_annotations_df.rename(columns={'target_id': 'round_id'}).set_index('round_id')

# get relevant info from human annotations
human_annotations_df = collapsed_df[['n_spans', 'span_text', 'span', 'label_set', 'unique_labels', 'full_text']].rename(columns={
    'n_spans': 'n_spans_human', 'span_text': 'span_texts_human', 'span': 'spans_human', 'label_set': 'cells_human', 'unique_labels': 'unique_cells_human'
})

# merge
merged_df = pd.merge(model_annotations_df, human_annotations_df, left_index=True, right_index=True)

merged_df.head()

,model_response,reasoning,error_type,error_msg,status_code,json_response,n_spans,spans,span_texts,cells,unique_cells,comments,no_reference,n_spans_human,span_texts_human,spans_human,cells_human,unique_cells_human,full_text
round_id,,,,,,,,,,,,,,,,,,,
1365-f75aa281-4887-4663-9826-b639f2440a6c_1,"{\n ""annotations"": [\n {\n ""span"": ""p...",None,None,None,None,{'annotations': [{'span': 'pale green in top m...,1,"[(9, 49)]",[pale green in top middle and middle left],"[[t2, t4]]","{t2, t4}",[],False,1,[pale green in top middle and middle left],"[(9, 49)]","[[t2, t4]]","{t2, t4}",speaker: pale green in top middle and middle left
1365-f75aa281-4887-4663-9826-b639f2440a6c_2,"{\n ""annotations"": [\n {\n ""span"": ""s...",None,None,None,None,{'annotations': [{'span': 'sky blue in top rig...,1,"[(9, 30)]",[sky blue in top right],[[t3]],{t3},[],False,1,[sky blue in top right],"[(9, 30)]",[[t3]],{t3},speaker: sky blue in top right
1365-f75aa281-4887-4663-9826-b639f2440a6c_3,"{\n ""annotations"": [\n {\n ""span"": ""i...",None,None,None,None,{'annotations': [{'span': 'its the one with th...,1,"[(9, 42)]",[its the one with the redest center],"[[t5, d1_5, d2_5]]","{d1_5, t5, d2_5}",[],False,1,[its the one with the redest center],"[(9, 43)]","[[d1_5, d2_5, t5]]","{d1_5, d2_5, t5}",speaker: its the one with the redest center
1365-f75aa281-4887-4663-9826-b639f2440a6c_4,"{\n ""annotations"": [\n {\n ""span"": ""n...",None,None,None,None,{'annotations': [{'span': 'neon green in the m...,1,"[(9, 48)]",[neon green in the middle of bottom row],[[t8]],{t8},[],False,1,[neon green in the middle of bottom row],"[(9, 47)]",[[t8]],{t8},speaker: neon green in the middle of bottom row
1365-f75aa281-4887-4663-9826-b639f2440a6c_5,"{\n ""annotations"": [\n {\n ""span"": ""g...",None,None,None,None,{'annotations': [{'span': 'grey in middle top ...,1,"[(9, 45)]",[grey in middle top and middle bottom],"[[t2, t8]]","{t8, t2}",[],False,1,[grey in middle top and middle bottom],"[(9, 45)]","[[t2, t8]]","{t8, t2}",speaker: grey in middle top and middle bottom


In [9]:
# number of entries that were annotated as "no reference" by the model

merged_df.no_reference.value_counts()

no_reference
False    1469
Name: count, dtype: int64

In [10]:
# reduce to entries that were not annotated as "no reference" by the model
# (should be all entries for human annotations, but can be important when investigating annotated model descriptions)

_merged_df = merged_df[~merged_df.no_reference]

In [11]:
# accuracy: SAME NUMBER OF SPANS

n_spans_acc = (_merged_df.n_spans == _merged_df.n_spans_human).values.mean() * 100
print(n_spans_acc.round(1))

87.0


In [12]:
# accuracy: SAME SPANS (reported in paper)
spans_acc = (_merged_df.span_texts == _merged_df.span_texts_human).values.mean() * 100
print(spans_acc.round(1))

72.4


In [13]:
# accuracy: SAME CELLS (sorted, reported in paper)
cells_acc = (_merged_df.cells.map(sorted) == _merged_df.cells_human.map(sorted)).values.mean() * 100
print(cells_acc.round(1))

67.3


In [14]:
# accuracy: SAME UNIQUE CELLS
unique_cells_acc = (_merged_df.unique_cells == _merged_df.unique_cells_human).values.mean() * 100
print(unique_cells_acc.round(1))

78.1


In [15]:
# removing distractor cells (since we are only looking at target features)
no_distractors = lambda x: set([c for c in x if not c.startswith('d')])

_merged_df['nd_unique_cells'] = _merged_df['unique_cells'].map(no_distractors)
_merged_df['nd_unique_cells_human'] = _merged_df['unique_cells_human'].map(no_distractors)

In [16]:
# accuracy: SAME UNIQUE CELLS WITHOUT DISTRACTORS (reported in paper)
unique_cells_acc = (_merged_df.nd_unique_cells == _merged_df.nd_unique_cells_human).values.mean() * 100
print(unique_cells_acc.round(1))

90.6


In [17]:
_merged_df.sample(10)[['unique_cells', 'unique_cells_human']]

,unique_cells,unique_cells_human
round_id,,
5948-6739fbf7-ad1a-4a61-bcbb-03cd77eb7b66_2,"{t3, t1}","{t3, t1}"
7086-79b4b0a8-9d2c-4f84-920d-d788aaaf54d3_57,"{t5, t2}","{t5, t2}"
2982-6442cb70-1654-4cfc-b4d2-dac8c02ec939_29,{t6},{t6}
3763-d036c1f6-aeab-4a7a-a55c-4bfe7ca1463a_39,{t4},{t4}
9955-1cbd5506-e782-422f-84e7-70366d4f805c_40,"{t5, t1, t2}","{t5, t1, t2}"
3763-d036c1f6-aeab-4a7a-a55c-4bfe7ca1463a_8,{t3},{t3}
8454-2e0a9860-ef78-4574-b9aa-8591c6d5541f_34,{t5},{t5}
5207-e36d6a16-30b0-47b2-a969-743b0f0ffe1d_60,"{t7, t1, t4}","{t7, t1, t4}"
5948-6739fbf7-ad1a-4a61-bcbb-03cd77eb7b66_29,{t6},{t6}


In [18]:
_merged_df[_merged_df.nd_unique_cells != _merged_df.nd_unique_cells_human][['nd_unique_cells', 'nd_unique_cells_human']]

,nd_unique_cells,nd_unique_cells_human
round_id,,
1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_10,{},{t9}
1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_21,{},{t6}
1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_26,"{t3, t1, t7}","{t3, t1}"
1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_27,"{t3, t5}","{t5, t1}"
1906-40c4b2ef-171b-4f0b-b70e-fb0955631406_19,"{t3, t5, t2, t9}","{t9, t2, t3, t1, t5}"
...,...,...
9955-1cbd5506-e782-422f-84e7-70366d4f805c_12,"{t4, t5, t7}","{t9, t5, t4}"
9955-1cbd5506-e782-422f-84e7-70366d4f805c_24,"{t8, t2, t4}",{t2}
9955-1cbd5506-e782-422f-84e7-70366d4f805c_34,"{t4, t1, t2, t7}","{t2, t1, t6, t7}"
